In [1]:
import csv

# read demand data
f = open("module2_quiz_P2a.csv")
csvfile = csv.DictReader(f)
headers = csvfile.fieldnames

table = []
for row in csvfile:
    table.append(row)
    
f.close()

Groups = []
Demand = {}
Months = range(1,6)
for row in table:
    Groups.append(row['Demand (tonne)'])
    for m in Months:
        Demand[(row['Demand (tonne)'], m)] = float(row[str("Month "+str(m))])

# read cost data
f = open("module2_quiz_P2b.csv")
csvfile = csv.DictReader(f)
headers = csvfile.fieldnames

table = []
for row in csvfile:
    table.append(row)
    
f.close()
                
Cost = {}
for row in table:
    for m in Months:
        Cost[(row['Production cost ($/tonne)'], m)] = float(row[str("Month "+str(m))])

In [2]:
ProcessingCapacity = 175
HoldingCost = 50
StorageCapacity = 50

In [3]:
from docplex.mp.model import Model
mdl = Model()

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


In [4]:
# variables
Processed = mdl.continuous_var_matrix(Groups, Months, lb=0, name='Processed')

In [5]:
# auxiliary variables: inventory at end of month
Inventory = mdl.continuous_var_matrix(Groups, Months, lb=0, ub=StorageCapacity, name='Inventory')

In [6]:
# objective: minimize cost
TotalProcessingCost = mdl.sum(Processed[(g,m)] * Cost[(g,m)] for g in Groups for m in Months)
TotalHoldingCost = mdl.sum(HoldingCost*Inventory[(g,m)] for g in Groups for m in Months)
mdl.minimize(TotalProcessingCost + TotalHoldingCost)

mdl.add_kpi(TotalProcessingCost, 'Total processing cost')
mdl.add_kpi(TotalHoldingCost, 'Total holding cost')

DecisionKPI(name=Total holding cost,expr=50Inventory_Large_1+50Inventory_Large_2+50Inventory_Large_3+50In..)

In [7]:
# total processing per month constraint - at most 175 tonnes of potatoes/month
for m in Months:
    mdl.add_constraint(mdl.sum(Processed[(g,m)] for g in Groups) <= ProcessingCapacity)

In [8]:
#demand per month per group constraint
#for m in Months:
#    for g in Groups:
#        mdl.add_constraint(Processed[(g,m)] >= Demand[(g,m)])
             

In [9]:
for m in range(6):
    for g in Groups:
        if m == 0:
            if g == 'Medium':
                Inventory[(g, m)] = 5.3
            else:
                Inventory[(g, m)] = 0

In [10]:
### inventory balance constraints
#for m in Months:
#    for g in Groups:
#        mdl.add_constraint(Inventory[(g,m)] == Inventory[(g,m-1)]*0.96 + Processed[(g,m)] - Demand [(g,m)])
        

In [11]:
### inventory balance constraints
for m in Months:
    for g in Groups:
        mdl.add_constraint(Inventory[(g,m)] == Inventory[(g,m-1)]*0.96 + Processed[(g,m)] - Demand [(g,m)])

In [12]:
# solve 
mdl.solve()
mdl.get_solve_details()

docplex.mp.SolveDetails(time=0.00215816,status='optimal')

In [13]:
mdl.print_solution()

objective: 255056.352
status: OPTIMAL_SOLUTION(2)
  Processed_Large_1=31.000
  Processed_Large_2=25.000
  Processed_Large_3=45.000
  Processed_Large_4=59.000
  Processed_Large_5=44.000
  Processed_Medium_1=44.912
  Processed_Medium_2=41.000
  Processed_Medium_3=70.000
  Processed_Medium_4=80.000
  Processed_Medium_5=60.000
  Processed_Small_1=42.000
  Processed_Small_2=63.854
  Processed_Small_3=60.000
  Processed_Small_4=36.000
  Processed_Small_5=56.000
  Inventory_Small_2=33.854
  Inventory_Small_3=37.500


In [14]:
mdl.report()

* model docplex_model1 solved with objective = 255056.352
*  KPI: Total processing cost = 251488.644
*  KPI: Total holding cost    = 3567.708
